# Sprint 4 — Deep Learning : MLP de base

**AquaSense AI · EHTP MIG S4 · CPU local ou Colab GPU**

Objectifs :
- Préprocessing DL (OneHot + MinMax + LabelEncoder)
- MLP Keras avec BatchNorm / Dropout
- Courbes d'apprentissage + métriques multi-classes
- Sauvegarde `models/mlp_best_v1.keras`

In [ ]:
# Cellule 1 — Google Drive + dézip (Colab uniquement)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cellule 2 — Dézipper + packages (Colab)
# Chemin Drive : ajuste si besoin

ZIP = "/content/drive/MyDrive/AquaSense_S4_Colab.zip"
!unzip -o -q "{ZIP}" -d /content/

# Colab a déjà TensorFlow, pandas, numpy, sklearn, matplotlib.
# Pas de venv. On installe seulement ce qui manque pour src.train (optionnel S4).
%pip install -q imbalanced-learn

%cd /content/AquaSense_S4_Colab
!ls -la data/cleaned/

In [ ]:
# Cellule 3 — Imports (après setup Colab)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path("/content/AquaSense_S4_Colab")  # Colab
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("..").resolve()              # local

sys.path.insert(0, str(PROJECT_ROOT))

from src.dl_utils import (
    MODELS_DIR,
    build_mlp,
    compile_model,
    evaluate_keras_model,
    load_clean_train,
    prepare_dl_data,
    split_xy,
    train_keras_model,
)
from src.train import CLASS_ORDER, NEEDS_REPAIR

sns.set_theme(style="whitegrid")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT =", PROJECT_ROOT)

## 1. Chargement & séparation X / y

In [ ]:
df = load_clean_train()
X_raw, y_raw = split_xy(df)

print("Shape X :", X_raw.shape)
print("Distribution cible :")
print(y_raw.value_counts(normalize=True).mul(100).round(1))

le = LabelEncoder()
le.fit(CLASS_ORDER)
print("\nEncodage y : 0=functional, 1=needs repair, 2=non-functional")

## 2. Préprocessing DL

In [ ]:
data = prepare_dl_data(test_size=0.2)
print("X_train :", data["X_train"].shape)
print("X_val   :", data["X_val"].shape)
print("Class weights :", data["class_weight"])

## 3. Architecture & entraînement MLP

In [ ]:
mlp = build_mlp(data["input_dim"], n_classes=3, dropout=0.3)
compile_model(mlp, lr=1e-3, class_weight=data["class_weight"])
mlp.summary()

history = train_keras_model(
    mlp,
    data,
    epochs=100,
    batch_size=512,
    validation_split=0.2,
    use_class_weight=True,
)

## 4. Courbes loss / accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Évaluation & sauvegarde

In [ ]:
metrics = evaluate_keras_model(mlp, data["X_val"], data["y_val"], data["label_encoder"])

print(f"F1-Macro        : {metrics['f1_macro']:.4f}")
print(f"F1 needs repair : {metrics['f1_needs_repair']:.4f}")
print(f"Recall needs repair : {metrics['recall_needs_repair']:.4f}")
print(f"Accuracy        : {metrics['accuracy']:.4f}")

y_pred = metrics["y_pred"]
print("\n", classification_report(
    data["y_val"], y_pred,
    target_names=["functional", "needs repair", "non-functional"],
))

cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["functional", "needs repair", "non-functional"],
            yticklabels=["functional", "needs repair", "non-functional"], ax=ax)
ax.set_title("Matrice de confusion — MLP")
plt.tight_layout()
plt.show()

save_path = MODELS_DIR / "mlp_best_v1.keras"
mlp.save(save_path)
print(f"Modèle sauvegardé → {save_path}")